# NYC Taxi: 11.8 million points in browser Python

This notebook renders all January 2015 NYC Taxi pickup locations from the 385 MB HoloViz Parquet dataset. Upload or mount `nyc_taxi_wide.parq` in JupyterLite, or enable the download cell below. Parquet decoding and Datashader aggregation are bounded by the selected batch size; the complete table is never constructed as one DataFrame.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import pyarrow.parquet as pq
import datashader.transfer_functions as tf

from datashader_browser import StreamingCanvas, iter_parquet_batches

DATA_URL = 'https://datasets.holoviz.org/nyc_taxi/v1/nyc_taxi_wide.parq'
DATA_PATH = Path('nyc_taxi_wide.parq')
X_RANGE = (-8_256_000, -8_209_000)
Y_RANGE = (4_964_000, 4_990_000)

Set `DOWNLOAD = True` to fetch the public dataset. Mounting or uploading it through JupyterLite avoids repeating this 385 MB download on subsequent sessions.

In [ ]:
DOWNLOAD = False

if DOWNLOAD and not DATA_PATH.exists():
    with urlopen(DATA_URL) as response, DATA_PATH.open('wb') as output:
        while block := response.read(1024 * 1024):
            output.write(block)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} is missing. Upload/mount it in JupyterLite or set DOWNLOAD=True.'
    )

parquet = pq.ParquetFile(DATA_PATH)
print(f'{parquet.metadata.num_rows:,} rows; {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

In [ ]:
def report(progress):
    if progress.chunks % 10 == 0 or progress.rows == parquet.metadata.num_rows:
        print(f'aggregated {progress.rows:,}/{parquet.metadata.num_rows:,} rows')

canvas = StreamingCanvas(
    plot_width=1_200,
    plot_height=700,
    x_range=X_RANGE,
    y_range=Y_RANGE,
)
aggregate = canvas.points(
    iter_parquet_batches(
        DATA_PATH,
        batch_size=250_000,
        columns=['pickup_x', 'pickup_y'],
    ),
    'pickup_x',
    'pickup_y',
    on_progress=report,
)
aggregate.attrs

In [ ]:
assert aggregate.attrs['source_rows'] == parquet.metadata.num_rows
image = tf.shade(
    aggregate,
    cmap=['#111827', '#1d4ed8', '#06b6d4', '#facc15', '#fff7ed'],
    how='eq_hist',
)
tf.set_background(image, '#050814').to_pil()